# DARKEN Dissociated Neurons Reanalysis with CalciMetrics 2.0

Reanalysis of dissociated neuron recordings from the DARKEN sDARKEN characterisation dataset.

**Pipeline:**  
`raw CSV` → background subtraction → ΔF/F (median baseline) → inspect ROIs → exclude bad ROIs → save `_cleaned.csv` → combine all ROIs → linear detrend → Gaussian smooth (window=5, σ=2.5) → % ΔF/F

> **Section 0** covers the preprocessing steps that produce the cleaned CSVs (not needed in the slice notebook, where cleaned files already existed).  
> Sections 1 onward load those cleaned files and run the full analysis pipeline.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from calcimetrics import Dataset, load_imagej_csv
from calcimetrics.core import Recording
from calcimetrics.io import export_cleaned_csv
from calcimetrics.normalization import compute_dff, to_percent
from calcimetrics.detrending import detrend_linear
from calcimetrics.smoothing import gaussian_smooth
from calcimetrics.plotting import plot_overlay, plot_mean_with_error, plot_heatmap

In [ ]:
# ── Plot style ───────────────────────────────────────────────────────────
# Set USE_NATURE_STYLE = True for a publication-ready Nature-journal look.
# Requires SciencePlots:  pip install SciencePlots
# 'no-latex' means LaTeX does not need to be installed locally.
# Set to False to use the default matplotlib style.

USE_NATURE_STYLE = True

import matplotlib as mpl

def apply_style():
    """Apply (or revert) the chosen style. Call again after seaborn imports."""
    if USE_NATURE_STYLE:
        try:
            import scienceplots  # noqa: F401
            plt.style.use(["science", "nature", "no-latex"])
            # A few extra tweaks for larger, more legible figures in a notebook
            mpl.rcParams.update({
                "figure.dpi":      150,
                "font.size":       9,
                "axes.labelsize":  10,
                "axes.titlesize":  10,
                "xtick.labelsize": 8,
                "ytick.labelsize": 8,
                "legend.fontsize": 8,
                "lines.linewidth": 1.0,
            })
        except ImportError:
            print("SciencePlots not found — using default style.\n"
                  "Install with:  pip install SciencePlots")
    else:
        plt.style.use("default")

apply_style()
print(f"Style: {'science + nature + no-latex' if USE_NATURE_STYLE else 'default'}  "
      f"| dpi = {mpl.rcParams['figure.dpi']:.0f}")
# ── Figure export ────────────────────────────────────────────────────────
# Set SAVE_FIGURES = True to export every plot as PNG + SVG.
SAVE_FIGURES = True
FIGURES_DIR  = Path.cwd().parent / "figures" / "darken_diss"

if SAVE_FIGURES:
    FIGURES_DIR.mkdir(exist_ok=True)
    print(f"Figures will be saved to: {FIGURES_DIR}")

def save_fig(fig, name: str) -> None:
    """Save *fig* as <name>.png and <name>.svg inside FIGURES_DIR."""
    if not SAVE_FIGURES:
        return
    for ext in ("png", "svg"):
        fig.savefig(FIGURES_DIR / f"{name}.{ext}",
                    dpi=300, bbox_inches="tight")
    print(f"  saved  {name}.png / .svg")


## 0. Preprocessing: Raw CSV → Cleaned CSV

This section converts raw ImageJ exports into cleaned CSVs ready for analysis.  
It mirrors the workflow in `Information/DARKEN_characterization/725/imaging_analysis_darken_725.ipynb` but uses the CalciMetrics package.

**Steps:**
1. **Load** each raw CSV — `load_imagej_csv` automatically drops the frame-index column and subtracts the last column (background ROI).
2. **ΔF/F** — normalise to the median fluorescence per ROI.
3. **Inspect** — subplot grid with one trace per ROI, labelled by column name.
4. **Exclude** — fill in the `"exclude"` lists, then re-run the next two cells.
5. **Save** — writes `<original_name>_cleaned.csv` next to the raw file.

> Once the cleaned files exist you only need to run Sections 1–5; Section 0 can be skipped.

In [ ]:
# ── 0.1  Raw-data configuration ──────────────────────────────────────────
# Add one entry per recording. Point "file" to the raw ImageJ CSV.
# Leave "exclude" as [] on the first run, then fill in column names
# (e.g. ["Mean3", "Mean7"]) after inspecting the plots below.

data_root = Path.cwd().parent / "input_data" / "darken_diss"

raw_configs = {
    "E2037": {
        "file":    data_root / "E2037" / "Results_E2037.csv",
        "fps":     3,
        "bio_rep": "E2037",
        "exclude": [],
    },
    "E2043": {
        "file":    data_root / "E2043" / "Results_E2043.csv",
        "fps":     3,
        "bio_rep": "E2043",
        "exclude": [],
    },
    "E2047": {
        "file":    data_root / "E2047" / "Results_E2047.csv",
        "fps":     3,
        "bio_rep": "E2047",
        "exclude": [],
    },
    "E2049": {
        "file":    data_root / "E2049" / "Results_E2049.csv",
        "fps":     3,
        "bio_rep": "E2049",
        "exclude": [],
    },
    "E2050": {
        "file":    data_root / "E2050" / "Results_E2050.csv",
        "fps":     3,
        "bio_rep": "E2050",
        "exclude": [],
    },
    "E2052": {
        "file":    data_root / "E2052" / "Results_E2052.csv",
        "fps":     3,
        "bio_rep": "E2052",
        "exclude": ["Mean6", "Mean7", "Mean8"],
    },
}

In [ ]:
# ── 0.2  Load raw CSVs, compute ΔF/F, and inspect individual ROIs ────────
# Run this cell to see every ROI labelled by its column name.
# Note which columns look bad (artefacts, flat traces, etc.) and add
# their names to the "exclude" list in raw_configs above.

recs_raw = {}

for rec_id, cfg in raw_configs.items():
    rec = load_imagej_csv(
        cfg["file"],
        fps=cfg["fps"],
        name=rec_id,
        biological_replicate=cfg["bio_rep"],
        condition="sDARKEN_diss",
        last_col_is_background=True,   # last ImageJ column = background ROI
    )
    rec.traces = compute_dff(rec.traces, method="median")
    recs_raw[rec_id] = rec

    n     = rec.traces.shape[1]
    ncols = 5
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(ncols * 4, nrows * 2),
        sharex=True, sharey=True,
    )
    axes_flat = np.array(axes).flatten()

    for j, col in enumerate(rec.traces.columns):
        ax = axes_flat[j]
        ax.plot(rec.time_axis, rec.traces[col], lw=0.8)
        ax.axhline(0, color="k", ls="--", lw=0.5)
        ax.set_title(col, fontsize=8)
        ax.spines["right"].set_visible(False)
        ax.spines["top"].set_visible(False)

    for ax in axes_flat[n:]:   # hide empty subplot panels
        ax.set_visible(False)

    fig.suptitle(
        f"{rec_id}  —  {n} ROIs  (fps = {cfg['fps']})\n"
        "Inspect traces; add bad column names to 'exclude' in raw_configs",
        y=1.02, fontsize=10,
    )
    plt.tight_layout()
    plt.show()
    print(f"  {rec_id}: {n} ROIs — {list(rec.traces.columns)}\n")

In [ ]:
# ── 0.3  Exclude bad ROIs and save cleaned CSVs ──────────────────────────
# After filling in the "exclude" lists in raw_configs (cell 0.1), re-run
# cells 0.1 → 0.2 → this cell.  A *_cleaned.csv is written next to each
# raw file; those paths are what Section 1 ("configs") should point to.

for rec_id, cfg in raw_configs.items():
    rec = recs_raw[rec_id]
    available = list(rec.traces.columns)

    # Guard: only exclude columns that actually exist; warn about mismatches.
    valid   = [c for c in cfg["exclude"] if c in available]
    missing = [c for c in cfg["exclude"] if c not in available]
    if missing:
        print(f"  {rec_id}: WARNING — column(s) not found, skipping: {missing}")
        print(f"    Available columns: {available}")

    if valid:
        rec = rec.exclude_rois(valid)
        print(f"  {rec_id}: excluded {valid} → {rec.traces.shape[1]} ROIs remaining")
    else:
        print(f"  {rec_id}: no exclusions → {rec.traces.shape[1]} ROIs")

    raw_path = Path(cfg["file"])
    out_path = raw_path.parent / (raw_path.stem + "_cleaned.csv")
    export_cleaned_csv(rec, out_path, use_time_index=False)
    print(f"    saved → {out_path}\n")

## 1. Load recordings

Each entry points to the `_cleaned.csv` produced by Section 0 (background-subtracted, ΔF/F, bad ROIs already removed). No further exclusions or normalisation are applied here.

In [ ]:
data_root = Path("/Users/lutz/Documents/data_science_projects/mannheim_new/upload")

configs = {
    "E2037": {"file": data_root / "E2037" / "Results_E2037_cleaned.csv", "bio_rep": "E2037"},
    "E2043": {"file": data_root / "E2043" / "Results_E2043_cleaned.csv", "bio_rep": "E2043"},
    "E2047": {"file": data_root / "E2047" / "Results_E2047_cleaned.csv", "bio_rep": "E2047"},
    "E2049": {"file": data_root / "E2049" / "Results_E2049_cleaned.csv", "bio_rep": "E2049"},
    "E2050": {"file": data_root / "E2050" / "Results_E2050_cleaned.csv", "bio_rep": "E2050"},
    "E2052": {"file": data_root / "E2052" / "Results_E2052_cleaned.csv", "bio_rep": "E2052"},
}

In [ ]:
neurons = Dataset()

for rec_id, cfg in configs.items():
    rec = load_imagej_csv(
        cfg["file"],
        fps=3,
        name=rec_id,
        biological_replicate=cfg["bio_rep"],
        condition="sDARKEN_diss",
        last_col_is_background=False,   # cleaned CSV has no background column
        background_subtract=False,      # already background-subtracted in Section 0
    )
    # ΔF/F already applied in Section 0 — no compute_dff here
    neurons.add(rec)
    print(f"  {rec_id}: {rec.traces.shape[1]} ROIs — {list(rec.traces.columns)}")

print()
print(neurons.summary())

## 2. Combine, detrend, smooth

All ROIs from all 4 recordings are pooled into one DataFrame (`level="roi"` = 35 cells).

Processing matches the original pipeline:
1. Linear detrend (`scipy.signal.detrend`)
2. Gaussian smooth (window = 5 frames, σ = 2.5)
3. Convert to % ΔF/F

A temporary `Recording` is created from the processed DataFrame so all `calcimetrics.plotting` functions can be used directly.

In [ ]:
# Pool all ROIs across recordings.
all_rois = neurons.combine_traces(level="roi")

# Pipeline: linear detrend → Gaussian smooth → % ΔF/F
all_rois = detrend_linear(all_rois)
all_rois = gaussian_smooth(all_rois, window_size=5, sigma=2.5)
all_rois = to_percent(all_rois)

n_rois = all_rois.shape[1]

# Wrap the processed DataFrame in a Recording so the plotting helpers work.
all_rec = Recording(
    name=f"all neurons (n = {n_rois} ROIs)",
    raw=all_rois,
    traces=all_rois,
    fps=3,
    excluded_rois=[],
    biological_replicate="pooled",
    condition="sDARKEN_diss",
)

print(f"Shape: {all_rois.shape}  "
      f"Time: 0–{all_rec.duration_s:.0f} s  "
      f"Value range: {all_rois.values.min():.1f} – {all_rois.values.max():.1f} % ΔF/F")

## 3. Plots

Three figures that reproduce the outputs saved in `all/cleaned/`,  
now using the `calcimetrics.plotting` functions.

In [ ]:
# Plot 1: All individual ROIs overlaid — slices_all_individual_rois
fig = plot_overlay(all_rec, ylabel="% ΔF/F", alpha=0.5)
save_fig(fig, "01_all_rois_overlay")
plt.show()


In [ ]:
# Plot 2: Mean ± SD — slices_all_mean_SD
fig = plot_mean_with_error(all_rec, error="sd", ylabel="% ΔF/F")
save_fig(fig, "02_all_rois_mean_sd")
plt.show()


In [ ]:
# Plot 3: Heatmap ROI × time — slices_all_heatmap
fig = plot_heatmap(all_rec, cbar_label="% ΔF/F", figsize=(10, 8))
save_fig(fig, "03_all_rois_heatmap")
plt.show()


## 4. Per-recording plots

The grand mean across all recordings in Section 3 averages out individual cell activity.  
The plots below apply the same pipeline (detrend → smooth → % ΔF/F) to each recording separately and show all-ROI overlay and mean ± SD per recording.

In [ ]:
for rec_id, rec in neurons.recordings.items():
    # Apply the same processing as for the pooled data.
    traces = detrend_linear(rec.traces.copy())
    traces = gaussian_smooth(traces, window_size=5, sigma=2.5)
    traces = to_percent(traces)

    proc_rec = Recording(
        name=f"{rec_id}  ({traces.shape[1]} ROIs)",
        raw=traces,
        traces=traces,
        fps=rec.fps,
        excluded_rois=rec.excluded_rois,
        biological_replicate=rec.biological_replicate,
        condition=rec.condition,
    )

    fig = plot_overlay(proc_rec, ylabel="% ΔF/F", alpha=0.6)
    save_fig(fig, f"04_{rec_id}_overlay")
    plt.show()

    fig = plot_mean_with_error(proc_rec, error="sd", ylabel="% ΔF/F")
    save_fig(fig, f"05_{rec_id}_mean_sd")
    plt.show()

## 5. Event analysis

sDARKEN fluorescence **decreases** (darkens) when serotonin binds, so spontaneous serotonin release events appear as **downward deflections** (negative % ΔF/F).

The setup cell below detects those events for every ROI using a −2 SD threshold and a minimum inter-event spacing of 20 s. The five plot cells that follow each tell a different part of the story.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.signal import find_peaks
from scipy.stats import expon

FPS = 3
SD_THRESHOLD = 2.0   # events must dip ≥ 2 SD below the ROI mean
MIN_DIST_S   = 20    # minimum 20 s between consecutive events
MIN_DIST_F   = int(MIN_DIST_S * FPS)

# Re-build per-recording processed recordings and detect events.
proc_recs = {}
events    = {}   # {rec_id: {col: array_of_frame_indices}}

for rec_id, rec in neurons.recordings.items():
    tr = detrend_linear(rec.traces.copy())
    tr = gaussian_smooth(tr, window_size=5, sigma=2.5)
    tr = to_percent(tr)
    proc_recs[rec_id] = Recording(
        name=f"{rec_id}",
        raw=tr, traces=tr, fps=rec.fps,
        excluded_rois=rec.excluded_rois,
        biological_replicate=rec.biological_replicate,
        condition=rec.condition,
    )
    events[rec_id] = {}
    for col in tr.columns:
        s = tr[col]
        # height in negated signal that corresponds to ≥ SD_THRESHOLD*std below mean
        min_h = -s.mean() + SD_THRESHOLD * s.std()
        peaks, _ = find_peaks(-s.values, height=min_h, distance=MIN_DIST_F)
        events[rec_id][col] = peaks

total = sum(len(v) for d in events.values() for v in d.values())
print(f"Detected {total} events  (threshold: −{SD_THRESHOLD} SD, min spacing: {MIN_DIST_S} s)")
for rid in events:
    n = sum(len(v) for v in events[rid].values())
    print(f"  {rid}: {n} events across {len(events[rid])} ROIs")

# Re-apply style after seaborn (seaborn can reset rcParams)
apply_style()

In [ ]:
# ── Plot 5a: Event raster plot ───────────────────────────────────────────
colors    = plt.cm.Set2.colors
slice_ids = list(events.keys())
cmap_     = {rid: colors[i % len(colors)] for i, rid in enumerate(slice_ids)}

fig, ax = plt.subplots(figsize=(12, 6))

y_pos   = 0
yticks  = []
ylabels = []
slice_y_centers = {}

for rid in slice_ids:
    y_start = y_pos
    for col, frames in events[rid].items():
        t_events = frames / FPS
        ax.scatter(t_events, [y_pos] * len(t_events),
                   marker='|', s=120, linewidth=1.8, color=cmap_[rid])
        yticks.append(y_pos)
        ylabels.append(col)
        y_pos += 1
    slice_y_centers[rid] = (y_start + y_pos - 1) / 2
    ax.axhspan(y_start - 0.5, y_pos - 0.5,
               color=cmap_[rid], alpha=0.08, zorder=0)
    y_pos += 1   # gap between slices

# Slice labels on right y-axis
ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim())
ax2.set_yticks(list(slice_y_centers.values()))
ax2.set_yticklabels([f"Slice {rid}" for rid in slice_y_centers], fontsize=9)
ax2.spines['right'].set_visible(False)

ax.set_yticks(yticks)
ax.set_yticklabels(ylabels, fontsize=7)
ax.set_xlabel("Time (s)")
ax.set_title("Spontaneous serotonin release events — raster plot")
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.tight_layout()
save_fig(fig, "06_event_raster")
plt.show()


In [ ]:
# ── Plot 5b: Average event waveform (aligned to peak) ───────────────────
WINDOW_S = 30          # ± 30 s window around each event peak
WINDOW_F = int(WINDOW_S * FPS)   # in frames

all_waves = []
per_slice = {rid: [] for rid in events}

for rid in events:
    tr = proc_recs[rid].traces
    for col, frames in events[rid].items():
        s = tr[col].values
        for pf in frames:
            start, end = pf - WINDOW_F, pf + WINDOW_F + 1
            if start >= 0 and end <= len(s):
                all_waves.append(s[start:end])
                per_slice[rid].append(s[start:end])

if not all_waves:
    print("No complete waveforms found — try relaxing the threshold.")
else:
    t_wave = np.arange(-WINDOW_F, WINDOW_F + 1) / FPS
    mat    = np.array(all_waves)

    fig, ax = plt.subplots(figsize=(8, 5))

    # Individual waveforms in light grey
    for w in mat:
        ax.plot(t_wave, w, color='grey', alpha=0.12, lw=0.6)

    # Per-slice means
    colors_ = plt.cm.Set2.colors
    for i, (rid, waves) in enumerate(per_slice.items()):
        if waves:
            m = np.array(waves).mean(axis=0)
            ax.plot(t_wave, m, color=colors_[i], lw=1.5, alpha=0.8,
                    label=f"Slice {rid} (n={len(waves)})")

    # Grand mean ± SEM
    grand_mean = mat.mean(axis=0)
    grand_sem  = mat.std(axis=0) / np.sqrt(len(mat))
    ax.plot(t_wave, grand_mean, color='black', lw=2.5,
            label=f"Grand mean  (n = {len(mat)} events)")
    ax.fill_between(t_wave, grand_mean - grand_sem, grand_mean + grand_sem,
                    color='black', alpha=0.25)

    ax.axhline(0, color='k', ls='--', lw=0.6)
    ax.axvline(0, color='red', ls='--', lw=1.0, label='Event peak')
    ax.set_xlabel("Time relative to event peak (s)")
    ax.set_ylabel("% ΔF/F")
    ax.set_title("Average sDARKEN event waveform")
    ax.legend(frameon=False, fontsize=8)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    plt.tight_layout()
    save_fig(fig, "07_event_waveform")
    plt.show()
